This was my first test for forward selection

# Forward Selection

In [6]:
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    if (cwd / "data_manipulation").is_dir():
        return cwd
    if (cwd.parent / "data_manipulation").is_dir():
        return cwd.parent
    return cwd  # fallback

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [7]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from functools import partial

from data_manipulation.data_split import create_dataloader, DemandDataset
from data_manipulation.data_creation import create_data
from model.functions import pinball_loss, rmse, train, get_test_loss

In [8]:
class simple_net(nn.Module):
    def __init__(self,input_size, hidden_size=10, output_size=1):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [9]:
all_specs = [
    "one_hot_month",
    "one_hot_week",
    "one_hot_day_of_week",
    "one_hot_weekend",
    "circular_sin_cos_month",
    "circular_sin_cos_week",
    "circular_sin_cos_day_of_week",
    "7_day_rolling_mean",
    "30_day_rolling_mean",
    "90_day_rolling_mean",
    "180_day_rolling_mean",
    "365_day_rolling_mean",
    "7_day_rolling_volatility",
    "30_day_rolling_volatility",
    "90_day_rolling_volatility",
    "180_day_rolling_volatility",
    "365_day_rolling_volatility",
    "7_day_rolling_min",
    "30_day_rolling_min",
    "90_day_rolling_min",
    "180_day_rolling_min",
    "365_day_rolling_min",
    "7_day_rolling_ema",
    "30_day_rolling_ema",
    "90_day_rolling_ema",
    "180_day_rolling_ema",
    "365_day_rolling_ema",
    "1_day_lag",
    "2_day_lag",
    "3_day_lag",
    "4_day_lag",
    "5_day_lag",
    "6_day_lag",
    "7_day_lag",
    "14_day_lag",
    "28_day_lag",
    "365_day_lag",
    "diff_1_day",
    "diff_7_day",
    "diff_30_day",
    "diff_90_day",
    "diff_180_day",
    "diff_365_day"
]

In [10]:
# Parameters
h_cost = 1
l_cost = 3
epochs = 300

# Initialize variables
best_loss = float("inf")
best_spec = None
all_losses = []

for spec in all_specs:

    # Get data for spec
    train_loader, val_loader, test_loader = create_dataloader(
        batch_size=32, 
        test_batch_size=32,
        pin_memory=False,
        data_mask=[
            ("store", 1),
            ("item", 1)
        ],
        specs=spec,
        )
    
    # Create model
    net = simple_net(train_loader.dataset.x.shape[1])
    loss = partial(pinball_loss, h_cost=h_cost, l_cost=l_cost)
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    # Train model
    _, _ = train(net, optimizer, loss, train_loader, val_loader, epochs=epochs, eval_interval=None, device="cpu")

    # Get test loss
    test_loss = get_test_loss(net, test_loader, loss, "cpu")

    # See how well it perfromed
    curr_loss = torch.tensor(test_loss).mean()

    # Save the best model
    if curr_loss < best_loss:
        best_loss = curr_loss
        best_spec = spec

    # Save all losses
    all_losses.append([spec, curr_loss])

    # Print the current loss
    print(f"Spec: {spec}, Loss: {curr_loss}")

Training: 100%|██████████| 300/300 [00:05<00:00, 57.27step/s, train_loss=21.4209]


Spec: one_hot_month, Loss: 26.334810256958008


Training: 100%|██████████| 300/300 [00:05<00:00, 58.51step/s, train_loss=18.3003]


Spec: one_hot_week, Loss: 30.399194717407227


Training: 100%|██████████| 300/300 [00:05<00:00, 55.48step/s, train_loss=13.2728]


...

Spec: diff_90_day, Loss: 28.27023696899414


Training: 100%|██████████| 300/300 [00:01<00:00, 209.28step/s, train_loss=17.4040]


Spec: diff_180_day, Loss: 22.768278121948242


Training: 100%|██████████| 300/300 [00:01<00:00, 226.00step/s, train_loss=17.9101]


Spec: diff_365_day, Loss: 29.25384521484375


In [11]:
print(f"Best spec: {best_spec} with loss {best_loss}")

Best spec: 7_day_rolling_ema with loss 5.922784328460693


In [12]:
# Print the 5 best specs
print("5 best specs:")
for spec, loss in sorted(all_losses, key=lambda x: x[1])[:5]:
    print(f"{spec}: {loss}")

5 best specs:
7_day_rolling_ema: 5.922784328460693
7_day_rolling_mean: 7.0457682609558105
30_day_rolling_ema: 7.302474498748779
90_day_rolling_ema: 8.193110466003418
14_day_lag: 8.5512056350708


Second Parameter

In [16]:
# Parameters
h_cost = 1
l_cost = 3
epochs = 300

# Initialize variables
best_loss = float("inf")
best_spec = None
all_losses = []

for spec in all_specs:
    if spec == "7_day_rolling_ema":
        continue

    # Get data for spec
    train_loader, val_loader, test_loader = create_dataloader(
        batch_size=32, 
        test_batch_size=32,
        pin_memory=False,
        data_mask=[
            ("store", 1),
            ("item", 1)
        ],
        specs=[spec] + ["7_day_rolling_ema"],
        )
    
    # Create model
    net = simple_net(train_loader.dataset.x.shape[1])
    loss = partial(pinball_loss, h_cost=h_cost, l_cost=l_cost)
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)

    # Train model
    _, _ = train(net, optimizer, loss, train_loader, val_loader, epochs=epochs, eval_interval=None, device="cpu")

    # Get test loss
    test_loss = get_test_loss(net, test_loader, loss, "cpu")

    # See how well it perfromed
    curr_loss = torch.tensor(test_loss).mean()

    # Save the best model
    if curr_loss < best_loss:
        best_loss = curr_loss
        best_spec = spec

    # Save all losses
    all_losses.append([spec, curr_loss])

    # Print the current loss
    print(f"Spec: {spec}, Loss: {curr_loss}")

Training: 100%|██████████| 300/300 [00:01<00:00, 197.95step/s, train_loss=6.0791] 


Spec: one_hot_month, Loss: 6.001124858856201


Training: 100%|██████████| 300/300 [00:01<00:00, 269.08step/s, train_loss=4.5053] 


Spec: one_hot_week, Loss: 5.948944091796875


Training: 100%|██████████| 300/300 [00:01<00:00, 283.82step/s, train_loss=4.3290] 


...

Spec: diff_90_day, Loss: 5.5050048828125


Training: 100%|██████████| 300/300 [00:01<00:00, 215.71step/s, train_loss=4.1927] 


Spec: diff_180_day, Loss: 5.031027317047119


Training: 100%|██████████| 300/300 [00:01<00:00, 231.58step/s, train_loss=3.9486] 


Spec: diff_365_day, Loss: 4.460465908050537


In [17]:
# Print the 5 best specs
print("5 best specs:")
for spec, loss in sorted(all_losses, key=lambda x: x[1])[:5]:
    print(f"{spec}: {loss}")

5 best specs:
diff_1_day: 3.6371328830718994
diff_30_day: 4.267825603485107
diff_365_day: 4.460465908050537
diff_180_day: 5.031027317047119
diff_7_day: 5.1149001121521


# L1 Regularization